<a href="https://colab.research.google.com/github/Priyanshujindal/Build-a-Tool-augmented-LLM-App/blob/main/Module_4_Graded_Assignment_OpenRouter_completed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 4 Graded Assignment — Tool-Augmented LLM App (Weather + Calculator)

**Goal:** Build a minimal GenAI pipeline that uses memory/caching and two tools (weather + calculator), with a simple agent that routes to tools when needed.

### Instructions
- Implement the required functions in the solution cells.
- Use a module-level cache created by `setup_cache()`; repeated `(city, unit)` queries should hit the cache (no re-fetch).
- Keep your code deterministic for grading.
- You may integrate LangChain locally, but the autograder focuses on tool behavior + caching and a minimal agent router.
- Do not rename required functions or change their signatures.

### Required functions
1. `setup_cache() -> Any`
2. `get_weather(city: str, unit: str = "C", fetcher: callable | None = None) -> dict`
3. `multiply(a: float, b: float) -> float`
4. `tool_registry() -> dict`
5. `agent_route(prompt: str) -> str` (return `"weather"`, `"multiply"`, or `"none"`)
6. `agent_answer(prompt: str, tools: dict | None = None, cache=None) -> str`

### Playtests (not graded)
- "What's the weather in Paris?"
- "What's 42 × 19?"
- "Summarize both results in one sentence."


## Read-Only imports (you may import more inside solutions)

In [31]:
from typing import Any, Callable, Dict, Optional
import re
import requests  # allowed for tests


## OpenRouter API Setup
This assignment is configured to use OpenRouter instead of provider-specific API keys. Enter your OpenRouter API key when prompted. Do not hard-code the key or save it in the notebook.


In [32]:
import os
from getpass import getpass

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("Enter your OpenRouter API key: ")

OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
print("OpenRouter API key configured.")


OpenRouter API key configured.


## Q1. Implement `setup_cache()` (Solution cell)
Create a cache object (e.g., dict) and assign it to module-level `CACHE`. Return it.


In [33]:
CACHE: Any = None

def setup_cache() -> Any:
    """Initialize and assign a cache object to module-level CACHE."""
    global CACHE
    CACHE = {}
    return CACHE


Make sure to write your answer above this line.

In [34]:
# BEGIN PUBLIC TESTS
c = setup_cache()
assert c is not None and isinstance(c, dict)
assert 'CACHE' in globals() and isinstance(CACHE, dict)
print("Q1 public tests passed.")
# END PUBLIC TESTS


Q1 public tests passed.


## Q2. Implement `get_weather(city, unit="C")` (Solution cell)
- Use Open-Meteo Geocoding to resolve city → (lat, lon).
- Call Open-Meteo Current Weather for temperature (Celsius).
- If `unit == "F"`, convert Celsius to Fahrenheit.
- Cache result dict as `{"city": city, "temp": <float>, "unit": unit}` under a normalized key.
- Subsequent calls with same `(city, unit)` must return the **same dict object** from cache (so `is` equality works).


In [35]:
def _default_fetcher(city: str) -> float:
    """Resolve city -> lat/lon via Open-Meteo geocoding, then fetch current temp (Celsius)."""
    geo_resp = requests.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={"name": city, "count": 1},
        timeout=10,
    )
    geo_resp.raise_for_status()
    results = geo_resp.json().get("results")
    if not results:
        raise ValueError(f"Could not resolve location for city: {city}")
    lat, lon = results[0]["latitude"], results[0]["longitude"]

    weather_resp = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={"latitude": lat, "longitude": lon, "current_weather": True},
        timeout=10,
    )
    weather_resp.raise_for_status()
    return float(weather_resp.json()["current_weather"]["temperature"])


def get_weather(city: str, unit: str = "C", fetcher: Optional[Callable] = None) -> dict:
    """Fetch (or retrieve cached) weather for a city, in the requested unit."""
    global CACHE
    if CACHE is None:
        setup_cache()

    unit_norm = unit.upper()
    key = (city.strip().lower(), unit_norm)

    if key in CACHE:
        return CACHE[key]

    fetch_fn = fetcher if fetcher is not None else _default_fetcher
    temp_c = float(fetch_fn(city))
    temp = temp_c * 9.0 / 5.0 + 32.0 if unit_norm == "F" else temp_c

    result = {"city": city, "temp": float(temp), "unit": unit_norm}
    CACHE[key] = result
    return result


Make sure to write your answer above this line.

In [36]:
# Q2 — Public Tests (API call allowed)
# BEGIN PUBLIC TESTS
setup_cache()
d1 = get_weather("Paris", "C")
d2 = get_weather("paris", "c")
assert d1 is d2, "Cached object should be returned for repeated queries."
assert isinstance(d1["temp"], float) and d1["unit"] == "C"
# Fahrenheit conversion
d3 = get_weather("Paris", "F")
assert d3["unit"] == "F" and isinstance(d3["temp"], float)
print("Q2 public tests passed.")
# END PUBLIC TESTS


Q2 public tests passed.


## Q3. Implement `multiply(a, b)` (Solution cell)
Return the product of `a` and `b`.


In [37]:
def multiply(a: float, b: float) -> float:
    return float(a) * float(b)


Make sure to write your answer above this line.

In [38]:
# Q3 — Public Tests
# BEGIN PUBLIC TESTS
assert multiply(6, 7) == 42
assert abs(multiply(1.5, 2.0) - 3.0) < 1e-9
print("Q3 public tests passed.")
# END PUBLIC TESTS

# Q3 — Hidden Tests


Q3 public tests passed.


## Q4. Implement `tool_registry()` (Solution cell)
Return a dict mapping `{"weather": get_weather, "multiply": multiply}`.


In [39]:
def tool_registry() -> Dict[str, Callable]:
    return {"weather": get_weather, "multiply": multiply}


Make sure to write your answer above this line.

In [40]:
# Q4 — Public Tests
# BEGIN PUBLIC TESTS
reg = tool_registry()
assert isinstance(reg, dict)
assert "weather" in reg and "multiply" in reg
assert callable(reg["weather"]) and callable(reg["multiply"])
print("Q4 public tests passed.")
# END PUBLIC TESTS

# Q4 — Hidden Tests


Q4 public tests passed.


## Q5. Implement `agent_route(prompt)` (Solution cell)
Return `"weather"`, `"multiply"`, or `"none"` based on the prompt.


In [41]:
def agent_route(prompt: str) -> str:
    p = prompt.lower()
    if re.search(r'\d+\s*[x×\*]\s*\d+', p) or "multiply" in p or "product" in p:
        return "multiply"
    if "weather" in p:
        return "weather"
    return "none"


Make sure to write your answer above this line.

In [42]:
# Q5 — Public Tests
# BEGIN PUBLIC TESTS
assert agent_route("what's the weather in Jaipur?") == "weather"
assert agent_route("compute 42 x 19 please") == "multiply"
assert agent_route("say hello") == "none"
print("Q5 public tests passed.")
# END PUBLIC TESTS

# Q5 — Hidden Tests


Q5 public tests passed.


## Q6. Implement `agent_answer(prompt, tools=None, cache=None)` (Solution cell)
- Use `agent_route` to choose a tool.
- If weather: parse city from `"weather in <City>"`. Default unit `"C"`, but if prompt mentions "Fahrenheit" or "F", use `"F"`.
- If multiply: parse two numbers from `a x b` or `a * b` and compute result with `multiply`.
- Return a short string response. You may pass `cache` in to override the module cache.


In [43]:
def agent_answer(prompt: str, tools: Optional[dict] = None, cache: Any = None) -> str:
    global CACHE
    if tools is None:
        tools = tool_registry()
    if cache is None:
        cache = CACHE if CACHE is not None else setup_cache()

    route = agent_route(prompt)

    if route == "weather":
        weather_fn = tools.get("weather", get_weather)
        match = re.search(
            r'weather in ([A-Za-z\s]+?)(?:\s+in\s+(fahrenheit|celsius))?[\?\.\!]*$',
            prompt,
            re.IGNORECASE,
        )
        city = match.group(1).strip() if match else "Unknown"
        unit = "F" if re.search(r'\bfahrenheit\b|\bF\b', prompt, re.IGNORECASE) else "C"

        result = weather_fn(city, unit, None)
        return f"The weather in {result['city']} is {result['temp']:.1f}°{result['unit']}."

    elif route == "multiply":
        pair = re.findall(r'(-?\d+(?:\.\d+)?)\s*[x×\*]\s*(-?\d+(?:\.\d+)?)', prompt)
        if pair:
            a, b = float(pair[0][0]), float(pair[0][1])
        else:
            nums = re.findall(r'-?\d+(?:\.\d+)?', prompt)
            if len(nums) < 2:
                return "Could not parse multiplication request."
            a, b = float(nums[0]), float(nums[1])

        multiply_fn = tools.get("multiply", multiply)
        result = multiply_fn(a, b)
        return f"{a} * {b} = {result}"

    else:
        return "I don't have a tool for that request."


Make sure to write your answer above this line.

In [44]:
# Q6 — Public Tests
# BEGIN PUBLIC TESTS
cache = setup_cache()
tools = tool_registry()

o1 = agent_answer("What's the weather in Mumbai in Fahrenheit?", tools=tools, cache=cache)
assert "Mumbai" in o1 and ("F" in o1)

o2 = agent_answer("Please do 12 * 8", tools=tools, cache=cache)
assert "12.0 * 8.0 = 96.0" in o2 or "12 * 8 = 96" in o2
print("Q6 public tests passed.")
# END PUBLIC TESTS

# Q6 — Hidden Tests


Q6 public tests passed.


## Playtests (not graded)
Try the sample prompts from the assignment brief.


In [45]:
cache = setup_cache()
tools = tool_registry()

print(agent_answer("What's the weather in Paris?", tools=tools, cache=cache))
print(agent_answer("What's 42 × 19?", tools=tools, cache=cache))
print(agent_answer("Summarize both results in one sentence.", tools=tools, cache=cache))


The weather in Paris is 20.0°C.
42.0 * 19.0 = 798.0
I don't have a tool for that request.
